# 龟龟选股器 (Turtle Screener)

**两层架构批量筛选器**：从全A股~5000只中，用策略所有可量化指标快速过滤候选标的。

- **Tier 1**：批量粗筛（~5秒，2个API调用） → 市场类指标过滤
- **Tier 2**：逐股深度分析（~5秒/股） → 财务质量 + 因子2/4 + 底价

In [ ]:
# Cell 1: 环境初始化
import sys
import os

# Add scripts to path
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), 'scripts'))
sys.path.insert(0, os.path.abspath('../scripts'))

from screener_config import ScreenerConfig
from screener_core import TushareScreener

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# 中文字体设置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ===== 配置参数（可自定义修改）=====
config = ScreenerConfig(
    min_listing_years=3,
    min_market_cap_yi=5.0,
    min_turnover_pct=0.1,
    max_pb=10.0,
    max_pe=50.0,
    obs_channel_limit=50,
    tier2_main_limit=150,
    min_roe=8.0,
    min_gross_margin=15.0,
    max_debt_ratio=70.0,
)

screener = TushareScreener(config=config)

# 显示配置
cfg_df = pd.DataFrame([
    {'参数': k, '值': v}
    for k, v in config.to_dict().items()
    if not k.startswith('cache')
])
display(cfg_df.style.hide(axis='index').set_caption('筛选器配置'))

In [ ]:
# Cell 2: Tier 1 批量粗筛
bulk_df = screener._tier1_bulk_data()
print(f'全A股数量: {len(bulk_df)}')

filtered = screener._tier1_filter(bulk_df)
main = filtered[filtered['channel'] == 'main']
obs = filtered[filtered['channel'] == 'observation']

print(f'\nTier 1 筛选结果:')
print(f'  通过筛选: {len(filtered)} 只')
print(f'  主通道: {len(main)} 只 (PE > 0 且 <= {config.max_pe})')
print(f'  观察通道: {len(obs)} 只 (PE < 0, 按市值取前{config.obs_channel_limit})')

# 行业分布饼图
if not filtered.empty and 'industry' in filtered.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 行业分布
    top_industries = filtered['industry'].value_counts().head(10)
    top_industries.plot.pie(ax=axes[0], autopct='%1.1f%%')
    axes[0].set_title('Tier 1 行业分布 (Top 10)')
    axes[0].set_ylabel('')
    
    # 市值分布
    (filtered['total_mv'] / 10000).plot.hist(ax=axes[1], bins=30, edgecolor='black')
    axes[1].set_title('市值分布 (亿元)')
    axes[1].set_xlabel('市值 (亿元)')
    axes[1].set_ylabel('数量')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 3: Tier 1 排序与截断
ranked = screener._tier1_rank_and_cut(filtered)
print(f'进入 Tier 2: {len(ranked)} 只')

# 展示候选列表
display_cols = ['ts_code', 'name', 'industry', 'close', 'pe_ttm', 'pb',
                'dv_ttm', 'turnover_rate', 'tier1_score', 'channel']
display_cols = [c for c in display_cols if c in ranked.columns]
display(ranked[display_cols].head(20).style
        .format({'pe_ttm': '{:.1f}', 'pb': '{:.2f}', 'dv_ttm': '{:.2f}',
                 'tier1_score': '{:.3f}', 'turnover_rate': '{:.2f}'},
                na_rep='—')
        .set_caption(f'Tier 1 候选 Top 20 / {len(ranked)}'))

# PE/PB/股息率 散点图
if not ranked.empty:
    main_r = ranked[ranked['channel'] == 'main']
    if not main_r.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        scatter = ax.scatter(main_r['pe_ttm'], main_r['pb'],
                           c=main_r['dv_ttm'], cmap='RdYlGn',
                           s=40, alpha=0.6)
        plt.colorbar(scatter, label='股息率 (%)')
        ax.set_xlabel('PE (TTM)')
        ax.set_ylabel('PB')
        ax.set_title('Tier 1 候选: PE vs PB (颜色=股息率)')
        plt.tight_layout()
        plt.show()

In [ ]:
# Cell 4: Tier 2 逐股深度分析
# 可调整 tier2_limit 控制分析数量 (None = 全部, 5 = 快速测试)
TIER2_LIMIT = 50  # 修改此值: None=全部200只, 5=快速测试

try:
    from tqdm.notebook import tqdm
    pbar = tqdm(total=min(TIER2_LIMIT or len(ranked), len(ranked)),
                desc='Tier 2 分析')
    def _progress(current, total, ts_code):
        pbar.n = current
        pbar.total = total
        pbar.set_postfix(stock=ts_code)
        pbar.refresh()
    
    result = screener.run(tier2_limit=TIER2_LIMIT, progress_callback=_progress)
    pbar.close()
except ImportError:
    result = screener.run(tier2_limit=TIER2_LIMIT)

print(f'\nTier 2 通过: {len(result)} 只')

In [ ]:
# Cell 5: 综合评分与排名
if not result.empty:
    display_cols = ['ts_code', 'name', 'industry', 'close', 'pe_ttm', 'pb',
                    'dv_ttm', 'roe_waa', 'gross_margin', 'fcf_yield', 'R',
                    'ev_ebitda', 'floor_premium', 'composite_score']
    display_cols = [c for c in display_cols if c in result.columns]
    
    top50 = result[display_cols].head(50)
    
    styled = (top50.style
        .format({
            'pe_ttm': '{:.1f}', 'pb': '{:.2f}', 'dv_ttm': '{:.2f}',
            'roe_waa': '{:.1f}%', 'gross_margin': '{:.1f}%',
            'fcf_yield': '{:.2f}%', 'R': '{:.2f}%',
            'ev_ebitda': '{:.1f}x', 'floor_premium': '{:.1f}%',
            'composite_score': '{:.3f}'
        }, na_rep='—')
        .background_gradient(subset=['composite_score'], cmap='RdYlGn')
        .set_caption(f'综合排名 Top {len(top50)}'))
    display(styled)
    
    # 综合得分分布
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    result['composite_score'].plot.hist(ax=axes[0], bins=20, edgecolor='black')
    axes[0].set_title('综合得分分布')
    axes[0].set_xlabel('Composite Score')
    
    # Top 10 行业分布
    if 'industry' in result.columns:
        result.head(20)['industry'].value_counts().plot.bar(ax=axes[1])
        axes[1].set_title('Top 20 行业分布')
        axes[1].set_ylabel('数量')
    
    plt.tight_layout()
    plt.show()
else:
    print('无结果')

In [ ]:
# Cell 6: 结果导出
if not result.empty:
    os.makedirs('../output', exist_ok=True)
    
    # CSV
    csv_path = '../output/screener_results.csv'
    screener.export_csv(result, csv_path)
    
    # HTML
    html_path = '../output/screener_results.html'
    screener.export_html(result, html_path)
    
    # Excel (optional)
    try:
        xlsx_path = '../output/screener_results.xlsx'
        result.to_excel(xlsx_path, index=False)
        print(f'Excel: {xlsx_path}')
    except Exception as e:
        print(f'Excel export skipped: {e}')
    
    print(f'\n共导出 {len(result)} 只股票')
else:
    print('无数据可导出')

In [ ]:
# Cell 7 (可选): 单股深入查看
# 输入股票代码，查看该股详细指标及在全样本中的百分位位置

STOCK_CODE = '600887.SH'  # 修改此处查看不同股票

if not result.empty and STOCK_CODE in result['ts_code'].values:
    stock = result[result['ts_code'] == STOCK_CODE].iloc[0]
    
    print(f"=== {stock['name']} ({STOCK_CODE}) ===")
    print(f"当前价格: {stock.get('close', '—')}")
    print(f"综合得分: {stock.get('composite_score', '—'):.3f}")
    rank = (result['ts_code'] == STOCK_CODE).values.argmax() + 1
    print(f"排名: {rank}/{len(result)}")
    print()
    
    metrics = [
        ('ROE(加权)', 'roe_waa', '%'),
        ('毛利率', 'gross_margin', '%'),
        ('FCF收益率', 'fcf_yield', '%'),
        ('穿透回报率R', 'R', '%'),
        ('EV/EBITDA', 'ev_ebitda', 'x'),
        ('底价溢价率', 'floor_premium', '%'),
        ('商誉/总资产', 'goodwill_ratio', '%'),
        ('净负债/EBITDA', 'net_debt_ebitda', 'x'),
    ]
    
    for label, col, unit in metrics:
        val = stock.get(col)
        if val is not None and val == val:
            pctile = (result[col] <= val).mean() * 100 if col in result.columns else None
            pctile_str = f" (百分位: {pctile:.0f}%)" if pctile is not None else ''
            print(f"  {label}: {val:.2f}{unit}{pctile_str}")
        else:
            print(f"  {label}: —")
else:
    if result.empty:
        print('请先运行 Cell 4-5')
    else:
        print(f'{STOCK_CODE} 不在结果中。可用股票:')
        print(result['ts_code'].head(10).tolist())